# Experimento 2: Clustering K-Medoids

Este notebook implementa el algoritmo K-Medoids sobre la matriz normalizada de características de trayectorias académicas. El modelo se evalúa mediante métricas internas de validación de clustering.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics import silhouette_score, davies_bouldin_score

from pyclustering.cluster.kmedoids import kmedoids

In [2]:
ruta = Path("../data/processed/X_clustering.csv")

X = pd.read_csv(ruta)

X.head()

,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
0,-0.487231,0.516412,-0.402032,-0.464573,-0.711080,-0.596566,0.147024,0.486179,0.433139,0.673603,-0.961629
1,-0.487231,-0.420977,-0.487768,0.028994,0.942910,1.321249,0.383717,-1.921609,-2.051139,-0.304771,1.363608
2,-0.487231,-0.366903,-0.487768,0.028994,0.777511,1.129467,0.265371,-1.921609,-2.009536,-0.222381,1.221827
3,-1.416016,-2.899670,-1.387992,-1.385899,0.446713,0.745904,-1.825423,-1.754733,-1.108143,-2.636562,1.369857
4,0.441553,0.189696,0.555349,0.950319,-0.545681,-0.596566,-0.629817,0.831851,1.050932,0.599907,-0.535305


In [3]:
X_array = X.values

In [4]:
resultados_kmedoids = []

for k in range(2, 11):

    # seleccionamos medoides iniciales aleatorios
    iniciales = list(range(k))

    modelo = kmedoids(
        X_array,
        iniciales
    )

    modelo.process()

    clusters = modelo.get_clusters()


    etiquetas = np.zeros(len(X_array), dtype=int)

    for i, cluster in enumerate(clusters):
        for punto in cluster:
            etiquetas[punto] = i


    silhouette = silhouette_score(
        X_array,
        etiquetas
    )

    davies = davies_bouldin_score(
        X_array,
        etiquetas
    )


    resultados_kmedoids.append({
        "k": k,
        "silhouette": silhouette,
        "davies_bouldin": davies
    })


resultados_kmedoids = pd.DataFrame(
    resultados_kmedoids
)

resultados_kmedoids

,k,silhouette,davies_bouldin
0,2,0.492114,0.897678
1,3,0.408611,0.896060
2,4,0.420435,1.156722
3,5,0.383823,1.293342
4,6,0.406134,1.112541
5,7,0.418663,0.983251
6,8,0.352345,1.029868
7,9,0.331437,1.061810
8,10,0.340558,1.004017


In [5]:

mejor_k = int(
    resultados_kmedoids
    .sort_values(
        ["silhouette", "davies_bouldin"],
        ascending=[False, True]
    )
    .iloc[0]["k"]
)

print("Mejor k seleccionado:", mejor_k)

Mejor k seleccionado: 2


In [6]:
modelo_final = kmedoids(
    X_array,
    list(range(mejor_k))
)

modelo_final.process()

clusters = modelo_final.get_clusters()


etiquetas_kmedoids = np.zeros(len(X_array), dtype=int)

for i, cluster in enumerate(clusters):
    for punto in cluster:
        etiquetas_kmedoids[punto] = i

In [7]:
pd.Series(etiquetas_kmedoids).value_counts()

0    172
1     72
Name: count, dtype: int64

In [8]:
df_original = pd.read_csv(
    "../data/processed/trayectorias_academicas.csv"
)

In [9]:
resultado_clusters = df_original.copy()

resultado_clusters["cluster"] = etiquetas_kmedoids


resultado_clusters.groupby(
    "cluster"
).mean(numeric_only=True)

,id_estudiante,periodos_registrados,promedio_global,creditos_totales,cursos_aprobados,cursos_desaprobados,cursos_previo_inicio,promedio_inicial,promedio_final,tendencia,tasa_aprobacion,variabilidad_nota
cluster,,,,,,,,,,,,
0,6.211051e+07,7.604651,14.588386,167.098837,56.075581,1.424419,0.773256,14.362707,14.988813,0.626106,0.978218,1.849505
1,6.155676e+07,4.111111,9.851589,92.666667,25.722222,11.166667,8.694444,13.210648,3.217007,-9.993641,0.649756,5.629196


In [10]:

silhouette_final = silhouette_score(
    X_array,
    etiquetas_kmedoids
)

davies_final = davies_bouldin_score(
    X_array,
    etiquetas_kmedoids
)

resultado_kmedoids = pd.DataFrame({
    "modelo": ["K-Medoids"],
    "configuracion": [f"k={mejor_k}"],
    "silhouette": [silhouette_final],
    "davies_bouldin": [davies_final]
})

resultado_kmedoids.to_csv(
    "../results/kmedoids/evaluacion_kmedoids.csv",
    index=False,
    encoding="utf-8-sig"
)

print(resultado_kmedoids.to_string(index=False))

   modelo configuracion  silhouette  davies_bouldin
K-Medoids           k=2    0.492114        0.897678


In [11]:
caracterizacion_kmedoids = (
    resultado_clusters
    .groupby("cluster")
    .mean(numeric_only=True)
)

caracterizacion_kmedoids.to_csv(
    "../results/kmedoids/caracterizacion_clusters.csv",
    encoding="utf-8-sig"
)

In [12]:

from pathlib import Path

# Conservar la posición de cada estudiante
asignaciones_kmedoids = pd.DataFrame({
    "fila": range(len(etiquetas_kmedoids)),
    "cluster": etiquetas_kmedoids
})

# Comprobar que las asignaciones corresponden a la matriz
assert len(asignaciones_kmedoids) == len(X)
assert len(asignaciones_kmedoids) == 244

ruta = Path("../results/kmedoids")
ruta.mkdir(parents=True, exist_ok=True)

asignaciones_kmedoids.to_csv(
    ruta / "asignaciones_kmedoids.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Asignaciones de K-Medoids guardadas correctamente")
print("Total de estudiantes:", len(asignaciones_kmedoids))

print("\nDistribución:")
print(
    asignaciones_kmedoids["cluster"]
    .value_counts()
    .sort_index()
)


Asignaciones de K-Medoids guardadas correctamente
Total de estudiantes: 244

Distribución:
cluster
0    172
1     72
Name: count, dtype: int64
